#$Vraj$ $Patel$

This is a simple project that predicts the next word.

We are using LSTM from pytorch

Language Modeling is the task of predicting the next word (or character) in a sequence based on the context of previous words.

In [1]:
!pip install nltk

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk

In [3]:
import re
text = open('text.txt', 'r').read().lower()

In [4]:
len(text)

85584

In [5]:
# Tokenization

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [6]:
#tokenize

tokens = word_tokenize(text.lower())

In [7]:
len(tokens)

16579

In [8]:
# build vocab

vocab = {'<UNK>':0}

Counter(tokens).keys()

dict_keys(['india', ',', 'officially', 'the', 'republic', 'of', '[', 'j', ']', '20', 'is', 'a', 'country', 'in', 'south', 'asia', '.', 'it', 'seventh-largest', 'by', 'area', ';', 'most', 'populous', 'since', '2023', '21', 'and', 'its', 'independence', '1947', 'world', "'s", 'democracy', '22', '23', '24', 'bounded', 'indian', 'ocean', 'on', 'arabian', 'sea', 'southwest', 'bay', 'bengal', 'southeast', 'shares', 'land', 'borders', 'with', 'pakistan', 'to', 'west', 'k', 'china', 'nepal', 'bhutan', 'north', 'bangladesh', 'myanmar', 'east', 'near', 'sri', 'lanka', 'maldives', 'andaman', 'nicobar', 'islands', 'share', 'maritime', 'border', 'thailand', 'indonesia', 'modern', 'humans', 'arrived', 'subcontinent', 'from', 'africa', 'no', 'later', 'than', '55,000', 'years', 'ago', '26', '27', '28', 'their', 'long', 'occupation', 'predominantly', 'isolation', 'as', 'hunter-gatherers', 'has', 'made', 'region', 'highly', 'diverse', '29', 'settled', 'life', 'emerged', 'western', 'margins', 'indus', 'r

In [9]:
for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token] = len(vocab)

In [10]:
vocab

{'<UNK>': 0,
 'india': 1,
 ',': 2,
 'officially': 3,
 'the': 4,
 'republic': 5,
 'of': 6,
 '[': 7,
 'j': 8,
 ']': 9,
 '20': 10,
 'is': 11,
 'a': 12,
 'country': 13,
 'in': 14,
 'south': 15,
 'asia': 16,
 '.': 17,
 'it': 18,
 'seventh-largest': 19,
 'by': 20,
 'area': 21,
 ';': 22,
 'most': 23,
 'populous': 24,
 'since': 25,
 '2023': 26,
 '21': 27,
 'and': 28,
 'its': 29,
 'independence': 30,
 '1947': 31,
 'world': 32,
 "'s": 33,
 'democracy': 34,
 '22': 35,
 '23': 36,
 '24': 37,
 'bounded': 38,
 'indian': 39,
 'ocean': 40,
 'on': 41,
 'arabian': 42,
 'sea': 43,
 'southwest': 44,
 'bay': 45,
 'bengal': 46,
 'southeast': 47,
 'shares': 48,
 'land': 49,
 'borders': 50,
 'with': 51,
 'pakistan': 52,
 'to': 53,
 'west': 54,
 'k': 55,
 'china': 56,
 'nepal': 57,
 'bhutan': 58,
 'north': 59,
 'bangladesh': 60,
 'myanmar': 61,
 'east': 62,
 'near': 63,
 'sri': 64,
 'lanka': 65,
 'maldives': 66,
 'andaman': 67,
 'nicobar': 68,
 'islands': 69,
 'share': 70,
 'maritime': 71,
 'border': 72,
 'thai

In [11]:
len(vocab)

3900

In [12]:
text = text.replace('\n','')

In [13]:
# extract sentences from data

input_sentences = text.split('.')

In [14]:
def text_to_indices(sentence, vocab):
  indexed_sentence = []
  for token in sentence:
    if token in vocab:
      indexed_sentence.append(vocab[token])
    else:
      indexed_sentence.append(vocab['<UNK>'])

  return indexed_sentence

In [15]:
input_sentences

['india, officially the republic of india,[j][20] is a country in south asia',
 " it is the seventh-largest country by area; the most populous country since 2023;[21] and, since its independence in 1947, the world's most populous democracy",
 '[22][23][24] bounded by the indian ocean on the south, the arabian sea on the southwest, and the bay of bengal on the southeast, it shares land borders with pakistan to the west;[k] china, nepal, and bhutan to the north; and bangladesh and myanmar to the east',
 ' in the indian ocean, india is near sri lanka and the maldives',
 ' its andaman and nicobar islands share a maritime border with myanmar, thailand, and indonesia',
 'modern humans arrived on the indian subcontinent from africa no later than 55,000 years ago',
 '[26][27][28] their long occupation, predominantly in isolation as hunter-gatherers, has made the region highly diverse',
 '[29] settled life emerged on the subcontinent in the western margins of the indus river basin 9,000 years a

In [16]:
input_sentences[0]

'india, officially the republic of india,[j][20] is a country in south asia'

In [17]:
input_numerical_sentences = []
for sentence in input_sentences:
  input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))

In [18]:
print(input_numerical_sentences[9])

[7, 131, 9, 7, 132, 9, 29, 133, 134, 4, 135, 136, 6, 137, 14, 1]


In [19]:
len(input_numerical_sentences)

685

In [20]:
training_seq = []

for sentence in input_numerical_sentences:

  for i in range(1, len(sentence)):
    training_seq.append(sentence[:i+1])

In [21]:
training_seq

[[1, 2],
 [1, 2, 3],
 [1, 2, 3, 4],
 [1, 2, 3, 4, 5],
 [1, 2, 3, 4, 5, 6],
 [1, 2, 3, 4, 5, 6, 1],
 [1, 2, 3, 4, 5, 6, 1, 2],
 [1, 2, 3, 4, 5, 6, 1, 2, 7],
 [1, 2, 3, 4, 5, 6, 1, 2, 7, 8],
 [1, 2, 3, 4, 5, 6, 1, 2, 7, 8, 9],
 [1, 2, 3, 4, 5, 6, 1, 2, 7, 8, 9, 7],
 [1, 2, 3, 4, 5, 6, 1, 2, 7, 8, 9, 7, 10],
 [1, 2, 3, 4, 5, 6, 1, 2, 7, 8, 9, 7, 10, 9],
 [1, 2, 3, 4, 5, 6, 1, 2, 7, 8, 9, 7, 10, 9, 11],
 [1, 2, 3, 4, 5, 6, 1, 2, 7, 8, 9, 7, 10, 9, 11, 12],
 [1, 2, 3, 4, 5, 6, 1, 2, 7, 8, 9, 7, 10, 9, 11, 12, 13],
 [1, 2, 3, 4, 5, 6, 1, 2, 7, 8, 9, 7, 10, 9, 11, 12, 13, 14],
 [1, 2, 3, 4, 5, 6, 1, 2, 7, 8, 9, 7, 10, 9, 11, 12, 13, 14, 15],
 [1, 2, 3, 4, 5, 6, 1, 2, 7, 8, 9, 7, 10, 9, 11, 12, 13, 14, 15, 16],
 [18, 11],
 [18, 11, 4],
 [18, 11, 4, 19],
 [18, 11, 4, 19, 13],
 [18, 11, 4, 19, 13, 20],
 [18, 11, 4, 19, 13, 20, 21],
 [18, 11, 4, 19, 13, 20, 21, 22],
 [18, 11, 4, 19, 13, 20, 21, 22, 4],
 [18, 11, 4, 19, 13, 20, 21, 22, 4, 23],
 [18, 11, 4, 19, 13, 20, 21, 22, 4, 23, 24],
 [18, 11,

In [22]:
training_seq[48]

[18,
 11,
 4,
 19,
 13,
 20,
 21,
 22,
 4,
 23,
 24,
 13,
 25,
 26,
 22,
 7,
 27,
 9,
 28,
 2,
 25,
 29,
 30,
 14,
 31,
 2,
 4,
 32,
 33,
 23,
 24]

In [23]:
len(training_seq)

15288

In [24]:
# since the length of all of these above sequences is not the same, we need to do padding.
# let's find the longest sequence in the training_seq

len_list = []

for sequence in training_seq:
  len_list.append(len(sentence))

max(len_list)

6

there's an error. somehow, the entire list contains 6 as the max number. let's debug

In [25]:
import random
for i in range(0,10):
  print(len(training_seq[random.randint(0,len(training_seq))]))

14
27
3
29
3
3
12
22
37
15


In [26]:
len(training_seq[10])

12

## I wrote sentence instead of sequence :)

In [27]:
# since the length of all of these above sequences is not the same, we need to do padding.
# let's find the longest sequence in the training_seq

len_list = []

for sequence in training_seq:
  len_list.append(len(sequence))

max(len_list)

139

We are simply padding 0 as 'pre'. atthe first places of the sentences.

In [28]:
padded_training_seq = []
for sequence in training_seq:
  padded_training_seq.append([0]*(max(len_list) - len(sequence)) + sequence)

In [29]:
len(padded_training_seq)

15288

In [30]:
padded_training_seq = torch.tensor(padded_training_seq, dtype=torch.long)

In [31]:
padded_training_seq

tensor([[   0,    0,    0,  ...,    0,    1,    2],
        [   0,    0,    0,  ...,    1,    2,    3],
        [   0,    0,    0,  ...,    2,    3,    4],
        ...,
        [   0,    0,    0,  ..., 3898,    9,    7],
        [   0,    0,    0,  ...,    9,    7, 3899],
        [   0,    0,    0,  ...,    7, 3899,    9]])

the last numbers are the target. so we will extract them.

In [32]:
padded_training_seq.shape

torch.Size([15288, 139])

In [33]:
X = padded_training_seq[:, :-1]
y = padded_training_seq[:,-1]

In [34]:
X

tensor([[   0,    0,    0,  ...,    0,    0,    1],
        [   0,    0,    0,  ...,    0,    1,    2],
        [   0,    0,    0,  ...,    1,    2,    3],
        ...,
        [   0,    0,    0,  ...,    7, 3898,    9],
        [   0,    0,    0,  ..., 3898,    9,    7],
        [   0,    0,    0,  ...,    9,    7, 3899]])

In [35]:
y

tensor([   2,    3,    4,  ...,    7, 3899,    9])

In [36]:
class CustomDataset(Dataset):

  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, index):
    return self.X[index], self.y[index]

In [37]:
dataset = CustomDataset(X, y)

In [38]:
len(dataset)

15288

In [39]:
dataset[0]

(tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]),
 tensor(2))

In [40]:
data_loader = DataLoader(dataset, batch_size = 32, shuffle = True)

In [41]:
for input, output in data_loader:
  print(input, output)

tensor([[   0,    0,    0,  ..., 3034, 3035,  366],
        [   0,    0,    0,  ...,    6,  181,   14],
        [   0,    0,    0,  ...,    4, 2805,    6],
        ...,
        [   0,    0,    0,  ...,    0,   29, 1161],
        [   0,    0,    0,  ...,    0,    0,    7],
        [   0,    0,    0,  ...,   39, 1047,   28]]) tensor([3036,    4,    4, 1601, 2817,  813, 2546,    7, 1936,   82,    9, 1298,
           4,    1,   32, 1938, 2763,    2, 2506,  240,    1,   20,   97, 1865,
           9,   26,   11, 3037,    6, 1820, 1594,  160])
tensor([[   0,    0,    0,  ...,    9,    7,   89],
        [   0,    0,    0,  ...,    2,  232,  105],
        [   0,    0,    0,  ...,  429, 3137,   14],
        ...,
        [   0,    0,    0,  ...,    4, 3134,    2],
        [   0,    0,    0,  ...,    0,    0,    7],
        [   0,    0,    0,  ..., 2826,   14,    0]]) tensor([   9,    2,    0,  245, 3148,    9,  128,    9, 3200, 3182,  815,    4,
          52,    1, 1298,  389,    6, 1873,   20,  

In [94]:
class LSTMModel(nn.Module):

  def __init__(self, vocab_size):

    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 100)
    self.lstm = nn.LSTM(100, 150, batch_first=True)
    self.fc = nn.Linear(150, vocab_size)

  def forward(self, X): # we will need to write the forward pass manually since lstm's have different inputs -> see below cells to understand it bettter
    embedded = self.embedding(X)
    intermediate_hidden_states, (final_hidden_state, final_cell_state) = self.lstm(embedded)
    output = self.fc(final_hidden_state.squeeze(0))

    return output

# why forward pass maunually?

In [95]:
x = nn.Embedding(len(vocab), 100)
y = nn.LSTM(100,150, batch_first=True)
z = nn.Linear(150, len(vocab))

In [96]:
dataset[343] # getting a random row

(tensor([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   7, 188]),
 tensor(9))

In [97]:
a = dataset[343][0].unsqueeze(0)

In [98]:
a

tensor([[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   7, 188]])

In [99]:
b = x(a)

In [100]:
c,d = y(b)

In [101]:
e,f = d

In [102]:
f.shape

torch.Size([1, 1, 150])

In [103]:
z(f.squeeze(0)).shape

torch.Size([1, 3900])

Now continue

In [104]:
vocab_size = len(vocab)

In [105]:
model = LSTMModel(vocab_size)

In [106]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

LSTMModel(
  (embedding): Embedding(3900, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=3900, bias=True)
)

In [107]:
epochs = 50
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [108]:
# training loop

for epoch in range(epochs):
  total_loss = 0

  for batch_x, batch_y in data_loader:

    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    optimizer.zero_grad()

    output = model(batch_x)

    loss = criterion(output, batch_y)

    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")

Epoch: 1, Loss: 3153.3840
Epoch: 2, Loss: 2744.6547
Epoch: 3, Loss: 2467.5990
Epoch: 4, Loss: 2169.6056
Epoch: 5, Loss: 1873.6007
Epoch: 6, Loss: 1597.9538
Epoch: 7, Loss: 1345.6988
Epoch: 8, Loss: 1124.5721
Epoch: 9, Loss: 935.7599
Epoch: 10, Loss: 777.2231
Epoch: 11, Loss: 644.7581
Epoch: 12, Loss: 534.7985
Epoch: 13, Loss: 444.0525
Epoch: 14, Loss: 370.8512
Epoch: 15, Loss: 309.1277
Epoch: 16, Loss: 261.5253
Epoch: 17, Loss: 223.7287
Epoch: 18, Loss: 194.2259
Epoch: 19, Loss: 170.9975
Epoch: 20, Loss: 153.5723
Epoch: 21, Loss: 141.3393
Epoch: 22, Loss: 130.4593
Epoch: 23, Loss: 123.2713
Epoch: 24, Loss: 117.9002
Epoch: 25, Loss: 112.7056
Epoch: 26, Loss: 109.5446
Epoch: 27, Loss: 107.8214
Epoch: 28, Loss: 107.2176
Epoch: 29, Loss: 107.7118
Epoch: 30, Loss: 105.8417
Epoch: 31, Loss: 100.9024
Epoch: 32, Loss: 98.5199
Epoch: 33, Loss: 97.0752
Epoch: 34, Loss: 96.2799
Epoch: 35, Loss: 95.5958
Epoch: 36, Loss: 94.8864
Epoch: 37, Loss: 94.9196
Epoch: 38, Loss: 99.8281
Epoch: 39, Loss: 143

Now Prediction

In [111]:
import torch

# prediction
def prediction(model, vocab, text):

  # tokenize
  tokenized_text = word_tokenize(text.lower())

  # text -> numerical indices
  numerical_text = text_to_indices(tokenized_text, vocab)

  # padding
  # Note: The original max_len was 139. Using 61 might cause issues if a sentence is longer.
  # For consistency, I will use the previously calculated max length of 139.
  max_len = 139 # from max(len_list) in cell djQhUbuPG_ki
  padded_text = torch.tensor([0] * (max_len - len(numerical_text)) + numerical_text, dtype=torch.long).unsqueeze(0)

  # Move padded_text to the same device as the model
  padded_text = padded_text.to(device)

  # send to model
  output = model(padded_text)

  # predicted index
  value, index = torch.max(output, dim=1)

  # merge with text
  # Ensure index is on CPU before converting to Python int for list indexing
  return text + " " + list(vocab.keys())[index.item()]

In [112]:
prediction(model, vocab, "capital of india is ")

'capital of india is  the'

In [115]:
import time

num_tokens = 30
input_text = "Delhi is a "

for i in range(num_tokens):
  output_text = prediction(model, vocab, input_text)
  print(output_text)
  input_text = output_text
  time.sleep(0.5)


Delhi is a  pluralistic
Delhi is a  pluralistic ,
Delhi is a  pluralistic , multilingual
Delhi is a  pluralistic , multilingual and
Delhi is a  pluralistic , multilingual and multi-ethnic
Delhi is a  pluralistic , multilingual and multi-ethnic society
Delhi is a  pluralistic , multilingual and multi-ethnic society ,
Delhi is a  pluralistic , multilingual and multi-ethnic society , a
Delhi is a  pluralistic , multilingual and multi-ethnic society , a griddled
Delhi is a  pluralistic , multilingual and multi-ethnic society , a griddled pancake
Delhi is a  pluralistic , multilingual and multi-ethnic society , a griddled pancake social
Delhi is a  pluralistic , multilingual and multi-ethnic society , a griddled pancake social reform
Delhi is a  pluralistic , multilingual and multi-ethnic society , a griddled pancake social reform ,
Delhi is a  pluralistic , multilingual and multi-ethnic society , a griddled pancake social reform , and
Delhi is a  pluralistic , multilingual and multi-ethnic

In [119]:
import time

num_tokens = 30
input_text = "A second urbanisation had taken"

for i in range(num_tokens):
  output_text = prediction(model, vocab, input_text)
  print(output_text)
  input_text = output_text
  time.sleep(0.5)

A second urbanisation had taken place
A second urbanisation had taken place in
A second urbanisation had taken place in 1991
A second urbanisation had taken place in 1991 ,
A second urbanisation had taken place in 1991 , when
A second urbanisation had taken place in 1991 , when the
A second urbanisation had taken place in 1991 , when the indian
A second urbanisation had taken place in 1991 , when the indian government
A second urbanisation had taken place in 1991 , when the indian government lists
A second urbanisation had taken place in 1991 , when the indian government lists the
A second urbanisation had taken place in 1991 , when the indian government lists the total
A second urbanisation had taken place in 1991 , when the indian government lists the total area
A second urbanisation had taken place in 1991 , when the indian government lists the total area as
A second urbanisation had taken place in 1991 , when the indian government lists the total area as 3,287,263
A second urbanisa

In [116]:
dataloader1 = DataLoader(dataset, batch_size=32, shuffle=False)

In [118]:
# Function to calculate accuracy
def calculate_accuracy(model, dataloader, device):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():  # No need to compute gradients
        for batch_x, batch_y in dataloader1:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            # Get model predictions
            outputs = model(batch_x)

            # Get the predicted word indices
            _, predicted = torch.max(outputs, dim=1)

            # Compare with actual labels
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    accuracy = correct / total * 100
    return accuracy

# Compute accuracy
accuracy = calculate_accuracy(model, data_loader, device)
print(f"Model Accuracy: {accuracy:.2f}%")


Model Accuracy: 96.13%
